In [ ]:
import torch

ckpt = torch.load(
    "/kaggle/input/models/maituananh511/resetnet-18/other/default/1/resnet18_chart_classifier_v2_best.pt",
    map_location="cpu"
)

print("Type:", type(ckpt))

if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys()))
    for k, v in ckpt.items():
        if not hasattr(v, 'shape'):
            print(f"  {k}: {v}")

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

MODEL_PATH = "/kaggle/input/models/maituananh511/resetnet-18/other/default/1/resnet18_chart_classifier_v2_best.pt"
DATA_DIR   = "/kaggle/input/datasets/maituananh511/test-classified/test_classified"
OUTPUT_DIR = "/kaggle/working"
BATCH_SIZE = 32
IMG_SIZE   = 224
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

dataset     = datasets.ImageFolder(DATA_DIR, transform=val_tf)
loader      = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
idx2class   = {v: k for k, v in dataset.class_to_idx.items()}
class_names = [idx2class[i] for i in range(len(idx2class))]
num_classes = len(class_names)

print(f"Dataset: {len(dataset)} anh | {num_classes} classes")
print(f"class_to_idx: {dataset.class_to_idx}")


class ResNetClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.resnet18(weights=None)
        self.resnet = base
        in_features = base.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Identity(),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)


ckpt        = torch.load(MODEL_PATH, map_location=DEVICE)
class_mapping = ckpt["class_mapping"]
idx2class_ckpt = {v: k for k, v in class_mapping.items()}
class_names    = [idx2class_ckpt[i] for i in range(len(idx2class_ckpt))]

folder_to_ckpt = {
    dataset.class_to_idx[cls]: class_mapping[cls]
    for cls in dataset.class_to_idx
}

model = ResNetClassifier(num_classes)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(DEVICE).eval()
print(f"Model loaded: {DEVICE}")
print(f"class_names: {class_names}")

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend([folder_to_ckpt[l.item()] for l in labels])

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
acc = (all_preds == all_labels).mean() * 100
print(f"Overall Accuracy: {acc:.2f}%")

cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

report_str  = classification_report(all_labels, all_preds,
                                    target_names=class_names, zero_division=0)
report_dict = classification_report(all_labels, all_preds,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
print(report_str)

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
with open(out / "classification_report.json", "w", encoding="utf-8") as f:
    json.dump(report_dict, f, ensure_ascii=False, indent=2)


def plot_confusion_matrix(cm_count, cm_pct, labels, save_path, overall_acc):
    n = len(labels)
    fig, ax = plt.subplots(figsize=(max(9, n * 1.15 + 2),
                                    max(8, n * 1.05 + 2)))
    fig.patch.set_facecolor("#FAFAFA")

    cmap = LinearSegmentedColormap.from_list(
        "wb", ["#FFFFFF", "#DBEAFE", "#3B82F6", "#1E3A8A"])

    sns.heatmap(
        cm_pct, ax=ax,
        cmap=cmap, vmin=0, vmax=1,
        linewidths=0.6, linecolor="#E2E8F0",
        annot=False,
        xticklabels=labels, yticklabels=labels,
        cbar_kws={"label": "row recall (%)", "shrink": 0.8},
    )

    for i in range(n):
        for j in range(n):
            cnt  = cm_count[i, j]
            pct  = cm_pct[i, j] * 100
            text = f"{cnt}\n{pct:.1f}%"
            fc   = "white" if cm_pct[i, j] > 0.55 else (
                   "#1E3A8A" if cm_pct[i, j] > 0.15 else "#64748B")
            ax.text(j + 0.5, i + 0.5, text,
                    ha="center", va="center",
                    fontsize=max(6.5, 10.5 - n // 4),
                    color=fc, fontweight="bold", linespacing=1.4)

    ax.set_xlabel("Predicted label", fontsize=13, labelpad=12, color="#334155")
    ax.set_ylabel("True label",      fontsize=13, labelpad=12, color="#334155")
    ax.set_title(
        f"Confusion Matrix  ·  Accuracy {overall_acc:.2f}%  ·  {n} classes",
        fontsize=15, pad=16, color="#0F172A", fontweight="normal")

    plt.xticks(rotation=40, ha="right",
               fontsize=max(8, 12 - n // 5), color="#475569")
    plt.yticks(rotation=0,
               fontsize=max(8, 12 - n // 5), color="#475569")

    cbar = ax.collections[0].colorbar
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cbar.set_ticklabels(["0%", "25%", "50%", "75%", "100%"])
    cbar.ax.tick_params(labelsize=9)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()

plot_confusion_matrix(cm, cm_norm, class_names,
                      out / "confusion_matrix_heatmap.png", acc)


def plot_per_class_metrics(report_dict, labels, save_path, overall_acc):
    prec = [report_dict[c]["precision"] * 100 for c in labels]
    rec  = [report_dict[c]["recall"]    * 100 for c in labels]
    f1   = [report_dict[c]["f1-score"]  * 100 for c in labels]

    x = np.arange(len(labels))
    w = 0.26
    fig, ax = plt.subplots(figsize=(max(10, len(labels) * 1.4 + 2), 6))
    fig.patch.set_facecolor("#FAFAFA")

    b1 = ax.bar(x - w, prec, w, label="Precision",
                color="#3B82F6", alpha=0.88, zorder=3)
    b2 = ax.bar(x,     rec,  w, label="Recall",
                color="#10B981", alpha=0.88, zorder=3)
    b3 = ax.bar(x + w, f1,   w, label="F1-score",
                color="#F59E0B", alpha=0.88, zorder=3)

    def add_labels(bars):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2,
                    h + 0.3, f"{h:.1f}",
                    ha="center", va="bottom",
                    fontsize=7.5, color="#374151", fontweight="bold")

    add_labels(b1); add_labels(b2); add_labels(b3)

    ax.axhline(overall_acc, color="#EF4444", linestyle="--",
               linewidth=1.3, zorder=4,
               label=f"Overall Acc {overall_acc:.1f}%")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right",
                       fontsize=10, color="#475569")
    ax.set_ylim(70, 106)
    ax.set_ylabel("Score (%)", fontsize=12, color="#334155")
    ax.set_title("Per-class Metrics  ·  Precision / Recall / F1",
                 fontsize=14, pad=14, color="#0F172A", fontweight="normal")

    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=10, colors="#475569")
    ax.legend(fontsize=10, loc="lower right",
              framealpha=0.9, edgecolor="#E2E8F0")

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()

plot_per_class_metrics(report_dict, class_names,
                       out / "per_class_metrics.png", acc)


def plot_summary_table(report_dict, labels, save_path, overall_acc):
    fig, ax = plt.subplots(figsize=(9, len(labels) * 0.55 + 2.5))
    fig.patch.set_facecolor("#FAFAFA")
    ax.axis("off")

    col_labels = ["Class", "Precision", "Recall", "F1-score", "Support"]
    rows = []
    for c in labels:
        m = report_dict[c]
        rows.append([c,
                     f"{m['precision']:.4f}",
                     f"{m['recall']:.4f}",
                     f"{m['f1-score']:.4f}",
                     str(int(m["support"]))])

    rows.append(["─" * 10] * 5)
    ma = report_dict["macro avg"]
    rows.append(["macro avg",
                 f"{ma['precision']:.4f}",
                 f"{ma['recall']:.4f}",
                 f"{ma['f1-score']:.4f}",
                 str(int(ma["support"]))])

    tbl = ax.table(cellText=rows, colLabels=col_labels,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10.5)
    tbl.scale(1, 1.55)

    for j in range(len(col_labels)):
        cell = tbl[0, j]
        cell.set_facecolor("#1E3A8A")
        cell.set_text_props(color="white", fontweight="bold")

    for i in range(1, len(rows) + 1):
        for j in range(len(col_labels)):
            cell = tbl[i, j]
            label = rows[i-1][0]
            if label.startswith("─"):
                cell.set_facecolor("#E2E8F0")
                cell.set_text_props(color="#E2E8F0")
                continue
            if label == "macro avg":
                cell.set_facecolor("#DBEAFE")
                cell.set_text_props(color="#1E3A8A", fontweight="bold")
            else:
                cell.set_facecolor("#FFFFFF" if i % 2 == 0 else "#F8FAFC")
                cell.set_text_props(color="#0F172A")
                if j == 3:
                    try:
                        v = float(rows[i-1][3])
                        if v >= 0.96:
                            cell.set_facecolor("#D1FAE5")
                        elif v < 0.90:
                            cell.set_facecolor("#FEF3C7")
                    except:
                        pass

    ax.set_title(f"Classification Report  ·  Accuracy {overall_acc:.2f}%",
                 fontsize=13, pad=18, color="#0F172A", fontweight="normal")

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()

plot_summary_table(report_dict, class_names,
                   out / "summary_table.png", acc)

In [ ]:
import matplotlib.pyplot as plt

# Dữ liệu từ lịch sử huấn luyện
history = {
    'train_loss': [0.3463633420812611, 0.17182976212168094, 0.14575055785624697, 0.13485807871980832, 0.10534370962604142, 0.10709992017928917, 0.09095905097846937, 0.07732279038997275, 0.09305111668303166, 0.0833559130401452, 0.055178906441214357, 0.05274392216291168, 0.07743884243144847, 0.05489946859381577, 0.04611868977933974, 0.04484105814192997, 0.06205669549837856, 0.0490874039922094, 0.03529236076565662, 0.05135827739769132, 0.030135363997375167, 0.02850813683961509, 0.036160598711876, 0.031194238462439267, 0.031167344245506396, 0.023881492241848223, 0.022058540824939046, 0.018751285314366296, 0.01820820567398655, 0.015014936572337283, 0.015944579397189882, 0.016646950177175206, 0.011850914034974014, 0.012901005642903545, 0.01157436211764038, 0.014632984233955567, 0.012290531417267343, 0.008855901288829909, 0.008049163253318592, 0.007700752250799485, 0.007273328053421516, 0.007227042614710563, 0.006014598708422301, 0.0063385550086305815, 0.006285056669932403, 0.006242881938215751, 0.006267198968026326],
    'train_acc': [75.26686262376238, 84.99767945544555, 87.0049504950495, 88.00278465346534, 89.52274133663366, 89.85535272277228, 90.66754331683168, 91.55321782178218, 91.12391707920793, 91.58802599009901, 93.5527537128713, 93.90083539603961, 92.14495668316832, 93.80414603960396, 94.4113551980198, 94.79424504950495, 93.5836943069307, 94.45389851485149, 95.59096534653466, 94.67821782178218, 95.75340346534654, 96.33740717821782, 95.74953589108911, 96.08601485148515, 96.19043935643565, 96.8440594059406, 97.13412747524752, 97.40485767326733, 97.50928217821782, 97.71813118811882, 97.6871905940594, 97.58663366336634, 98.25572400990099, 98.1667698019802, 98.35241336633663, 98.10102103960396, 98.25572400990099, 98.56126237623762, 98.83586014851485, 98.77784653465346, 98.78171410891089, 98.89387376237623, 98.9480198019802, 99.0253712871287, 99.07564975247524, 98.97509282178218, 99.02923886138613],
    'val_loss': [0.8152925589909921, 0.7253883664424603, 0.4704276862052771, 0.5946369549402823, 0.4443714412359091, 0.30550034086291605, 0.30392466943997604, 0.30760814306827694, 0.7348717898130417, 0.3091080736082334, 0.29202098886554057, 0.3678785516665532, 0.42168182077316135, 0.32704048345868403, 0.35269886140639967, 0.40176040965777177, 0.31823542857399356, 0.2679858204837029, 0.22317883974084488, 0.2599118478023089, 0.22325045386186013, 0.2734763663118848, 0.22816731178989777, 0.3405544829483216, 0.25081981174074686, 0.25766492599191576, 0.2761776281090883, 0.2745187978188579, 0.2656458716552991, 0.2634838775803263, 0.384627154813363, 0.295837204473523, 0.32150410760480624, 0.3320351571847613, 0.2772985246175757, 0.29791569982010585, 0.32527583321699727, 0.32645611407665104, 0.3472866132282294, 0.34663954921639883, 0.34690409540556943, 0.33699495159089565, 0.349746011197567, 0.32703937439677805, 0.3360119787259744, 0.32875450578733134, 0.33536201894569856],
    'val_acc': [59.29477265697495, 72.00742344571606, 76.15218063717909, 67.80080420661923, 76.49242189916487, 85.58614290133004, 83.66841942468295, 84.78193628209094, 76.05938756572843, 86.2356944014847, 87.25641818744201, 83.69935044849984, 83.94679863903495, 88.74110733065264, 86.69965975873801, 83.91586761521806, 85.24590163934427, 89.94741725951128, 90.25672749768017, 87.96783173523043, 91.03000309310238, 90.84441695020105, 89.97834828332817, 87.72038354469532, 91.12279616455305, 91.89607175997526, 88.33900402103309, 92.17445097432726, 93.07145066501701, 92.88586452211568, 86.79245283018868, 92.85493349829879, 91.43210640272193, 93.1023816888339, 93.19517476028456, 93.65914011753789, 93.628209093721, 94.33962264150944, 94.24682957005876, 94.21589854624187, 94.21589854624187, 94.24682957005876, 94.61800185586142, 94.37055366532633, 94.33962264150944, 94.61800185586142, 94.80358799876277],
    'per_class_acc': [{'area': 77.04918032786885, 'bar': 76.82926829268293, 'box': 86.36363636363636, 'heatmap': 92.87469287469287, 'histogram': 93.0, 'line': 28.615622583139984, 'pie': 96.3855421686747, 'scatter': 68.89952153110048}, {'area': 98.36065573770492, 'bar': 64.28571428571429, 'box': 73.86363636363636, 'heatmap': 88.45208845208845, 'histogram': 90.0, 'line': 69.21887084300077, 'pie': 100.0, 'scatter': 64.75279106858054}, {'area': 88.52459016393442, 'bar': 89.02439024390245, 'box': 81.81818181818181, 'heatmap': 93.85749385749386, 'histogram': 98.0, 'line': 63.34106728538283, 'pie': 100.0, 'scatter': 70.65390749601276}, {'area': 83.60655737704919, 'bar': 91.98606271777004, 'box': 94.31818181818181, 'heatmap': 91.64619164619165, 'histogram': 92.0, 'line': 47.87316318638825, 'pie': 100.0, 'scatter': 57.89473684210526}, {'area': 98.36065573770492, 'bar': 90.59233449477352, 'box': 92.04545454545455, 'heatmap': 93.61179361179362, 'histogram': 86.0, 'line': 60.94354215003867, 'pie': 100.0, 'scatter': 75.59808612440192}, {'area': 91.80327868852459, 'bar': 86.06271777003484, 'box': 92.04545454545455, 'heatmap': 94.34889434889435, 'histogram': 96.0, 'line': 80.2784222737819, 'pie': 100.0, 'scatter': 85.32695374800637}, {'area': 98.36065573770492, 'bar': 88.67595818815332, 'box': 90.9090909090909, 'heatmap': 92.38329238329239, 'histogram': 93.0, 'line': 75.48337200309358, 'pie': 100.0, 'scatter': 84.21052631578948}, {'area': 91.80327868852459, 'bar': 90.06968641114983, 'box': 93.18181818181819, 'heatmap': 89.1891891891892, 'histogram': 93.0, 'line': 77.10750193348801, 'pie': 100.0, 'scatter': 87.71929824561404}, {'area': 95.08196721311475, 'bar': 42.68292682926829, 'box': 95.45454545454545, 'heatmap': 88.6977886977887, 'histogram': 95.0, 'line': 76.95282289249806, 'pie': 100.0, 'scatter': 85.8054226475279}, {'area': 85.24590163934427, 'bar': 89.7212543554007, 'box': 90.9090909090909, 'heatmap': 95.08599508599508, 'histogram': 98.0, 'line': 81.67053364269141, 'pie': 98.79518072289157, 'scatter': 82.61562998405104}, {'area': 93.44262295081967, 'bar': 95.47038327526133, 'box': 93.18181818181819, 'heatmap': 92.13759213759214, 'histogram': 93.0, 'line': 81.12915699922661, 'pie': 100.0, 'scatter': 85.16746411483254}, {'area': 98.36065573770492, 'bar': 89.37282229965157, 'box': 94.31818181818181, 'heatmap': 83.78378378378379, 'histogram': 91.0, 'line': 80.35576179427687, 'pie': 100.0, 'scatter': 79.10685805422648}, {'area': 93.44262295081967, 'bar': 88.67595818815332, 'box': 95.45454545454545, 'heatmap': 86.24078624078624, 'histogram': 91.0, 'line': 79.50502706883218, 'pie': 100.0, 'scatter': 81.49920255183413}, {'area': 90.1639344262295, 'bar': 91.46341463414635, 'box': 92.04545454545455, 'heatmap': 95.08599508599508, 'histogram': 94.0, 'line': 86.0015467904099, 'pie': 98.79518072289157, 'scatter': 85.00797448165869}, {'area': 93.44262295081967, 'bar': 92.50871080139373, 'box': 88.63636363636364, 'heatmap': 93.85749385749386, 'histogram': 91.0, 'line': 85.1508120649652, 'pie': 100.0, 'scatter': 76.55502392344498}, {'area': 96.72131147540983, 'bar': 89.37282229965157, 'box': 80.68181818181819, 'heatmap': 95.33169533169533, 'histogram': 93.0, 'line': 74.94199535962878, 'pie': 98.79518072289157, 'scatter': 85.8054226475279}, {'area': 95.08196721311475, 'bar': 83.44947735191637, 'box': 94.31818181818181, 'heatmap': 94.1031941031941, 'histogram': 95.0, 'line': 81.28383604021656, 'pie': 100.0, 'scatter': 83.5725677830941}, {'area': 95.08196721311475, 'bar': 96.16724738675958, 'box': 89.77272727272727, 'heatmap': 93.61179361179362, 'histogram': 91.0, 'line': 89.55916473317865, 'pie': 100.0, 'scatter': 80.70175438596492}, {'area': 96.72131147540983, 'bar': 92.50871080139373, 'box': 93.18181818181819, 'heatmap': 96.31449631449631, 'histogram': 97.0, 'line': 87.70301624129931, 'pie': 100.0, 'scatter': 86.1244019138756}, {'area': 98.36065573770492, 'bar': 93.37979094076655, 'box': 96.5909090909091, 'heatmap': 95.33169533169533, 'histogram': 93.0, 'line': 83.91337973704563, 'pie': 100.0, 'scatter': 81.97767145135566}, {'area': 95.08196721311475, 'bar': 93.55400696864112, 'box': 93.18181818181819, 'heatmap': 94.5945945945946, 'histogram': 96.0, 'line': 87.78035576179428, 'pie': 98.79518072289157, 'scatter': 90.59011164274322}, {'area': 88.52459016393442, 'bar': 94.42508710801394, 'box': 89.77272727272727, 'heatmap': 92.62899262899263, 'histogram': 95.0, 'line': 90.17788089713844, 'pie': 100.0, 'scatter': 86.28389154704944}, {'area': 98.36065573770492, 'bar': 91.11498257839722, 'box': 95.45454545454545, 'heatmap': 93.12039312039312, 'histogram': 95.0, 'line': 88.16705336426914, 'pie': 100.0, 'scatter': 86.92185007974481}, {'area': 90.1639344262295, 'bar': 91.11498257839722, 'box': 89.77272727272727, 'heatmap': 96.06879606879608, 'histogram': 93.0, 'line': 83.21732405259087, 'pie': 100.0, 'scatter': 85.48644338118022}, {'area': 96.72131147540983, 'bar': 93.37979094076655, 'box': 89.77272727272727, 'heatmap': 93.61179361179362, 'histogram': 95.0, 'line': 88.78576952822893, 'pie': 100.0, 'scatter': 90.1116427432217}, {'area': 91.80327868852459, 'bar': 95.1219512195122, 'box': 87.5, 'heatmap': 94.84029484029485, 'histogram': 94.0, 'line': 89.63650425367362, 'pie': 100.0, 'scatter': 90.9090909090909}, {'area': 98.36065573770492, 'bar': 94.42508710801394, 'box': 86.36363636363636, 'heatmap': 96.31449631449631, 'histogram': 94.0, 'line': 79.969064191802, 'pie': 100.0, 'scatter': 91.70653907496013}, {'area': 95.08196721311475, 'bar': 94.77351916376307, 'box': 87.5, 'heatmap': 93.61179361179362, 'histogram': 94.0, 'line': 90.8739365815932, 'pie': 100.0, 'scatter': 90.59011164274322}, {'area': 91.80327868852459, 'bar': 94.42508710801394, 'box': 88.63636363636364, 'heatmap': 93.85749385749386, 'histogram': 94.0, 'line': 92.73008507347255, 'pie': 100.0, 'scatter': 91.70653907496013}, {'area': 91.80327868852459, 'bar': 95.1219512195122, 'box': 92.04545454545455, 'heatmap': 93.36609336609337, 'histogram': 92.0, 'line': 92.5754060324826, 'pie': 100.0, 'scatter': 90.59011164274322}, {'area': 90.1639344262295, 'bar': 95.81881533101046, 'box': 86.36363636363636, 'heatmap': 94.84029484029485, 'histogram': 93.0, 'line': 76.87548337200309, 'pie': 100.0, 'scatter': 90.74960127591707}, {'area': 90.1639344262295, 'bar': 94.5993031358885, 'box': 89.77272727272727, 'heatmap': 95.57739557739558, 'histogram': 92.0, 'line': 91.26063418406805, 'pie': 98.79518072289157, 'scatter': 92.82296650717703}, {'area': 88.52459016393442, 'bar': 93.72822299651568, 'box': 88.63636363636364, 'heatmap': 95.33169533169533, 'histogram': 91.0, 'line': 88.8631090487239, 'pie': 98.79518072289157, 'scatter': 91.86602870813397}, {'area': 93.44262295081967, 'bar': 95.99303135888502, 'box': 85.22727272727273, 'heatmap': 94.5945945945946, 'histogram': 93.0, 'line': 93.03944315545243, 'pie': 98.79518072289157, 'scatter': 89.95215311004785}, {'area': 91.80327868852459, 'bar': 95.64459930313589, 'box': 88.63636363636364, 'heatmap': 95.08599508599508, 'histogram': 94.0, 'line': 93.96751740139212, 'pie': 100.0, 'scatter': 87.87878787878788}, {'area': 91.80327868852459, 'bar': 95.81881533101046, 'box': 86.36363636363636, 'heatmap': 94.84029484029485, 'histogram': 93.0, 'line': 93.81283836040217, 'pie': 98.79518072289157, 'scatter': 91.2280701754386}, {'area': 88.52459016393442, 'bar': 95.29616724738676, 'box': 87.5, 'heatmap': 94.5945945945946, 'histogram': 94.0, 'line': 94.04485692188709, 'pie': 98.79518072289157, 'scatter': 91.2280701754386}, {'area': 91.80327868852459, 'bar': 97.0383275261324, 'box': 85.22727272727273, 'heatmap': 94.34889434889435, 'histogram': 93.0, 'line': 94.04485692188709, 'pie': 100.0, 'scatter': 93.46092503987241}, {'area': 90.1639344262295, 'bar': 96.68989547038328, 'box': 86.36363636363636, 'heatmap': 94.84029484029485, 'histogram': 93.0, 'line': 93.96751740139212, 'pie': 100.0, 'scatter': 93.14194577352472}, {'area': 93.44262295081967, 'bar': 96.86411149825784, 'box': 87.5, 'heatmap': 94.34889434889435, 'histogram': 93.0, 'line': 93.58081979891725, 'pie': 98.79518072289157, 'scatter': 93.62041467304626}, {'area': 91.80327868852459, 'bar': 96.34146341463415, 'box': 87.5, 'heatmap': 94.34889434889435, 'histogram': 93.0, 'line': 94.43155452436194, 'pie': 98.79518072289157, 'scatter': 92.50398724082935}, {'area': 91.80327868852459, 'bar': 96.68989547038328, 'box': 88.63636363636364, 'heatmap': 95.08599508599508, 'histogram': 93.0, 'line': 94.04485692188709, 'pie': 98.79518072289157, 'scatter': 92.50398724082935}, {'area': 90.1639344262295, 'bar': 96.34146341463415, 'box': 88.63636363636364, 'heatmap': 94.84029484029485, 'histogram': 93.0, 'line': 95.20494972931168, 'pie': 98.79518072289157, 'scatter': 92.6634768740032}, {'area': 91.80327868852459, 'bar': 96.51567944250871, 'box': 88.63636363636364, 'heatmap': 95.08599508599508, 'histogram': 93.0, 'line': 94.81825212683681, 'pie': 98.79518072289157, 'scatter': 91.70653907496013}, {'area': 90.1639344262295, 'bar': 96.86411149825784, 'box': 88.63636363636364, 'heatmap': 95.33169533169533, 'histogram': 93.0, 'line': 94.27687548337201, 'pie': 98.79518072289157, 'scatter': 92.3444976076555}, {'area': 90.1639344262295, 'bar': 96.51567944250871, 'box': 89.77272727272727, 'heatmap': 95.08599508599508, 'histogram': 93.0, 'line': 95.28228924980665, 'pie': 98.79518072289157, 'scatter': 92.02551834130782}, {'area': 90.1639344262295, 'bar': 96.34146341463415, 'box': 89.77272727272727, 'heatmap': 95.08599508599508, 'histogram': 93.0, 'line': 95.51430781129157, 'pie': 98.79518072289157, 'scatter': 92.6634768740032}],
    'lr': [0.001, 0.0009990143508499217, 0.0009960612933065818, 0.00099115248173898, 0.0009843072889837512, 0.0009755527298894294, 0.0009649233547011816, 0.0009524611127067769, 0.0009382151866819099, 0.0009222417987882566, 0.0009046039886902864, 0.0008853713647665069, 0.0008646198293969952, 0.0008424312794113801, 0.0008188932828794706, 0.0007940987335200906, 0.0007681454840920089, 0.0007411359602138069, 0.0007131767561367538, 0.0006843782140659967, 0.0006548539886902863, 0.0006247205986388449, 0.0005940969666355696, 0.0005631039501653701, 0.0005318638645048921, 0.0005005, 0.00046913613549510796, 0.00043789604983463003, 0.00040690303336443054, 0.000376279401361155, 0.00034614601130971394, 0.00031662178593400354, 0.00028782324386324626, 0.00025986403978619317, 0.00023285451590799105, 0.0002069012664799097, 0.00018210671712052946, 0.00015856872058861996, 0.00013638017060300502, 0.0001156286352334933, 9.639601130971378e-05, 7.875820121174356e-05, 6.278481331809012e-05, 4.853888729322332e-05, 3.607664529881844e-05, 2.5447270110570804e-05, 1.6692711016248827e-05]
}

# 1. Plot Loss và Accuracy
epochs = range(1, len(history['train_loss']) + 1)

# Sử dụng subplots để vẽ
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs, history['train_loss'], label='Training Loss')
ax1.plot(epochs, history['val_loss'], label='Validation Loss')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(epochs, history['train_acc'], label='Training Accuracy')
ax2.plot(epochs, history['val_acc'], label='Validation Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('training_metrics.png')

# 2. Vẽ Per-Class Accuracy (Epoch cuối cùng)
final_class_acc = history['per_class_acc'][-1]
classes = list(final_class_acc.keys())
accuracies = list(final_class_acc.values())

fig2, ax3 = plt.subplots(figsize=(10, 6))
ax3.bar(classes, accuracies, color='skyblue')
ax3.set_title('Per-Class Accuracy (Final Epoch)')
ax3.set_ylabel('Accuracy (%)')
plt.savefig('per_class_accuracy.png')

# 3. Vẽ Learning Rate
fig3, ax4 = plt.subplots(figsize=(10, 4))
ax4.plot(epochs, history['lr'], color='green')
ax4.set_title('Learning Rate Schedule')
ax4.set_xlabel('Epochs')
ax4.set_ylabel('Learning Rate')
plt.savefig('learning_rate.png')